# DS483 – Natural Language Processing Project
## SMS Spam Detection: from classical NLP to Transformers

**Group members:** *(write names + IDs here)*  **CRN:** *(write CRN here)*

This notebook contains the complete implementation for the five project tasks.
All models are trained on the **same training split**, tuned on the **same validation data** (or cross-validation inside the training split), and evaluated **once** on the **same held-out test split**.

| Task | Content |
|---|---|
| 1 | EDA + text preprocessing pipeline |
| 2 | TF-IDF representation + cosine similarity |
| 3 | Naive Bayes & Logistic Regression |
| 4 | BiLSTM classifier (PyTorch) |
| 5 | Fine-tuned DistilBERT + final comparison |

**Run on Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU). Total run time is roughly 10 minutes.

In [ ]:
!pip -q install transformers nltk

In [ ]:
import os, re, io, zipfile, random, time, json, urllib.request, warnings
from collections import Counter
from functools import lru_cache
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

warnings.filterwarnings("ignore")
SEED = 42
RESULTS = {}                      # everything we need for the report is stored here
os.makedirs("figures", exist_ok=True)
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "serif"

## 0. Data loading and train / validation / test split

The SMS Spam Collection (UCI) contains 5,574 labelled messages. The raw file contains **exact duplicate messages**.
If a duplicate lands in both train and test, the test score is inflated (data leakage), so duplicates are removed **before** splitting (`DROP_DUPLICATES = True`).
The split is stratified 70 / 15 / 15 because the classes are imbalanced.

In [ ]:
DROP_DUPLICATES = True

def load_sms():
    fname = "SMSSpamCollection"
    if not os.path.exists(fname):
        url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
        try:
            data = urllib.request.urlopen(url, timeout=60).read()
            zipfile.ZipFile(io.BytesIO(data)).extract(fname)
        except Exception as e:
            raise RuntimeError("Download failed. Download the zip manually from the UCI page, extract the file "
                               "'SMSSpamCollection' and upload it next to this notebook.") from e
    for enc in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(fname, sep="\t", header=None, names=["label", "text"], quoting=3, encoding=enc)
        except UnicodeDecodeError:
            continue

raw = load_sms()
n_raw, n_dups = len(raw), int(raw.duplicated().sum())
df = raw.drop_duplicates().reset_index(drop=True) if DROP_DUPLICATES else raw.copy()
df["y"] = (df["label"] == "spam").astype(int)

train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df["y"], random_state=SEED)
val_df, test_df  = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df["y"], random_state=SEED)
train_df, val_df, test_df = [d.copy().reset_index(drop=True) for d in (train_df, val_df, test_df)]

RESULTS["data"] = dict(raw_messages=n_raw, exact_duplicates=n_dups, kept=len(df),
                       ham=int((df.y == 0).sum()), spam=int((df.y == 1).sum()),
                       train=len(train_df), val=len(val_df), test=len(test_df),
                       spam_ratio_train=round(float(train_df.y.mean()), 4),
                       spam_ratio_test=round(float(test_df.y.mean()), 4))
print(json.dumps(RESULTS["data"], indent=1))

# Task 1 – Text Preprocessing

### 1.1 Class distribution and textual characteristics

In [ ]:
print("Raw class counts (before duplicate removal):\n", raw["label"].value_counts().to_string(), "\n")
print("Class counts used in the project:\n", df["label"].value_counts().to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
sns.countplot(data=df, x="label", order=["ham", "spam"], ax=ax[0], palette=["#4C72B0", "#C44E52"])
ax[0].set_title("Class distribution"); ax[0].set_xlabel("")
for p in ax[0].patches:
    ax[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
df["n_chars"] = df.text.str.len()
sns.kdeplot(data=df, x="n_chars", hue="label", common_norm=False, fill=True, ax=ax[1], clip=(0, 400))
ax[1].set_title("Message length (characters)")
plt.tight_layout(); plt.savefig("figures/t1_distribution_length.png", dpi=200); plt.show()

In [ ]:
f = pd.DataFrame({"label": df.label})
f["chars"]        = df.text.str.len()
f["words"]        = df.text.str.split().str.len()
f["digits"]       = df.text.str.count(r"\d")
f["upper_ratio"]  = df.text.apply(lambda t: sum(c.isupper() for c in t) / max(len(t), 1))
f["has_url"]      = df.text.str.contains(r"(?:https?://|www\.|\.com|\.co\.uk)", case=False).astype(int)
f["has_money"]    = df.text.str.contains(r"[£$€]").astype(int)
f["has_long_num"] = df.text.str.contains(r"\b\d{5,}\b").astype(int)
f["exclaim"]      = df.text.str.count("!")
feat_table = f.groupby("label").mean(numeric_only=True).round(3).T
display(feat_table)
RESULTS["eda_feature_means"] = {c: feat_table[c].to_dict() for c in feat_table.columns}

def top_words(label, k=12):
    c = Counter(w for t in df[df.label == label].text for w in re.findall(r"[a-z]+", t.lower()) if w not in set(stopwords.words("english")))
    return c.most_common(k)
tw = {l: top_words(l) for l in ("ham", "spam")}
RESULTS["eda_top_words_no_stopwords"] = tw
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, l, col in zip(ax, ("ham", "spam"), ("#4C72B0", "#C44E52")):
    w, c = zip(*tw[l]); a.barh(w[::-1], c[::-1], color=col); a.set_title(f"Most frequent words – {l}")
plt.tight_layout(); plt.savefig("figures/t1_top_words.png", dpi=200); plt.show()

### 1.2 Preprocessing pipeline

Steps (in order):
1. **Regular-expression normalisation** – e-mails → `emailtok`, URLs → `urltok`, currency amounts → `moneytok`, long digit strings (phone numbers / short codes) → `phonetok`, other numbers → `numtok`. Specific values are rarely repeated, but the *presence* of a URL, a prize amount or a phone number is a strong spam signal.
2. **Case normalisation** (lower-casing).
3. **Punctuation removal / tokenisation** with a regular expression that keeps alphabetic tokens only.
4. **Stop-word removal** (NLTK English list) – *optional*.
5. **Stemming** (Porter) – *optional*.

Steps 4 and 5 are not assumed to help: SMS text is short and some stop-words (e.g. *you, your, now*) are typical of spam. They are decided by a small **ablation with 5-fold cross-validation on the training split only**.

In [ ]:
STOP = set(stopwords.words("english"))
_stemmer = PorterStemmer()
@lru_cache(maxsize=None)
def stem(w): return _stemmer.stem(w)

URL_RE   = re.compile(r"(?:https?://\S+|www\.\S+|\b[\w-]+\.(?:com|co\.uk|net|org|info|biz)\b\S*)", re.I)
EMAIL_RE = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
MONEY_RE = re.compile(r"[£$€]\s?\d[\d,]*(?:\.\d+)?|\b\d+(?:\.\d+)?\s?(?:p|pence|pounds?|gbp|usd)\b", re.I)
PHONE_RE = re.compile(r"\b\d{5,}\b")
NUM_RE   = re.compile(r"\b\d+\b")

def normalise(text):
    t = EMAIL_RE.sub(" emailtok ", text)
    t = URL_RE.sub(" urltok ", t)
    t = MONEY_RE.sub(" moneytok ", t)
    t = PHONE_RE.sub(" phonetok ", t)
    t = NUM_RE.sub(" numtok ", t)
    return t.lower()

def preprocess(text, stop=True, stem_words=True):
    toks = re.findall(r"[a-z]+", normalise(text))
    if stop:       toks = [w for w in toks if w not in STOP]
    if stem_words: toks = [stem(w) for w in toks]
    return " ".join(toks)

variants = {"lower + punctuation only":      dict(stop=False, stem_words=False),
            "+ stop-word removal":           dict(stop=True,  stem_words=False),
            "+ stemming":                    dict(stop=False, stem_words=True),
            "+ stop-words + stemming":       dict(stop=True,  stem_words=True)}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for name, cfg in variants.items():
    Xc = train_df.text.map(lambda t: preprocess(t, **cfg))
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
    s = cross_val_score(pipe, Xc, train_df.y, cv=cv, scoring="f1")
    vocab = len(set(w for t in Xc for w in t.split()))
    rows.append(dict(variant=name, vocab_size=vocab, cv_f1_mean=round(s.mean(), 4), cv_f1_std=round(s.std(), 4)))
abl = pd.DataFrame(rows); display(abl)
best_name = abl.sort_values(["cv_f1_mean", "vocab_size"], ascending=[False, True]).iloc[0]["variant"]
PREP = variants[best_name]
print("Selected preprocessing for the classical models:", best_name, PREP)
RESULTS["preprocessing_ablation"] = abl.to_dict("records"); RESULTS["preprocessing_selected"] = best_name

In [ ]:
for d in (train_df, val_df, test_df):
    d["clean"] = d.text.map(lambda t: preprocess(t, **PREP))

ex = pd.concat([train_df[train_df.y == 1].sample(3, random_state=1), train_df[train_df.y == 0].sample(3, random_state=1)])
ex_table = ex[["label", "text", "clean"]].rename(columns={"text": "original", "clean": "after preprocessing"})
display(ex_table)
RESULTS["preprocessing_examples"] = ex_table.to_dict("records")

# Task 2 – TF-IDF Representation and Cosine Similarity

The TF-IDF vectoriser is **fitted on the training messages only**. Cosine similarity between two TF-IDF vectors measures the angle between them, so it ignores message length and focuses on shared, discriminative terms.

In [ ]:
tfidf = TfidfVectorizer(sublinear_tf=True)
X_train = tfidf.fit_transform(train_df.clean)
X_test_t = tfidf.transform(test_df.clean)
terms = np.array(tfidf.get_feature_names_out())
print("TF-IDF matrix (train):", X_train.shape, "| density: %.4f%%" % (100 * X_train.nnz / np.prod(X_train.shape)))

# highest mean TF-IDF weight per class
top_terms = {}
for lab, name in ((1, "spam"), (0, "ham")):
    m = np.asarray(X_train[train_df.y.values == lab].mean(axis=0)).ravel()
    top_terms[name] = [(terms[i], round(float(m[i]), 4)) for i in np.argsort(-m)[:10]]
display(pd.DataFrame({k: [f"{w} ({s})" for w, s in v] for k, v in top_terms.items()}))
RESULTS["task2_top_tfidf_terms"] = top_terms

In [ ]:
def most_similar(query, k=5):
    q = tfidf.transform([preprocess(query, **PREP)])
    sims = cosine_similarity(q, X_train).ravel()
    idx = np.argsort(-sims)[:k]
    out = train_df.iloc[idx][["label", "text"]].copy(); out.insert(0, "cosine", sims[idx].round(3))
    return q, idx, out

queries = [test_df[test_df.y == 1].iloc[0].text, test_df[test_df.y == 1].iloc[1].text, test_df[test_df.y == 0].iloc[0].text]
RESULTS["task2_similarity_examples"] = []
for q_text in queries:
    q, idx, out = most_similar(q_text)
    print("\nQUERY:", q_text); display(out)
    contrib = q.multiply(X_train[idx[0]]).toarray().ravel()
    shared = [(terms[i], round(float(contrib[i]), 3)) for i in np.argsort(-contrib)[:5] if contrib[i] > 0]
    print("Terms contributing most to the top match:", shared)
    RESULTS["task2_similarity_examples"].append(dict(query=q_text, top=out.to_dict("records"), shared_terms=shared))

In [ ]:
# (a) within-class vs between-class similarity on a random sample (up to 300 messages per class)
rs = np.random.RandomState(SEED)
ns = min(300, int(train_df.y.sum()))
ham_i  = rs.choice(np.where(train_df.y == 0)[0], ns, replace=False)
spam_i = rs.choice(np.where(train_df.y == 1)[0], ns, replace=False)
def mean_sim(a, b, same):
    S = cosine_similarity(X_train[a], X_train[b])
    if same: S = S[~np.eye(len(a), dtype=bool)]
    return float(S.mean())
sim_stats = dict(ham_ham=mean_sim(ham_i, ham_i, True), spam_spam=mean_sim(spam_i, spam_i, True), ham_spam=mean_sim(ham_i, spam_i, False))
print({k: round(v, 4) for k, v in sim_stats.items()})

# (b) heat-map for 8 ham + 8 spam messages
sel = np.r_[ham_i[:8], spam_i[:8]]
plt.figure(figsize=(5.8, 5))
sns.heatmap(cosine_similarity(X_train[sel]), cmap="viridis", xticklabels=["H"]*8+["S"]*8, yticklabels=["H"]*8+["S"]*8, vmin=0, vmax=1)
plt.title("Cosine similarity (H = ham, S = spam)"); plt.tight_layout(); plt.savefig("figures/t2_similarity_heatmap.png", dpi=200); plt.show()

# (c) 1-nearest-neighbour classification with cosine similarity
nn = np.asarray(cosine_similarity(X_test_t, X_train).argmax(axis=1)).ravel()
nn_pred = train_df.y.values[nn]
nn_stats = dict(accuracy=float(accuracy_score(test_df.y, nn_pred)), precision=float(precision_score(test_df.y, nn_pred)),
                recall=float(recall_score(test_df.y, nn_pred)), f1=float(f1_score(test_df.y, nn_pred)))
print("1-NN (cosine) on test:", {k: round(v, 4) for k, v in nn_stats.items()})
RESULTS["task2_similarity_stats"] = sim_stats; RESULTS["task2_1nn_test"] = nn_stats

# Task 3 – Text Classification: Naive Bayes vs Logistic Regression

Both classifiers sit on top of a TF-IDF vectoriser inside an sklearn `Pipeline`, so the vectoriser is re-fitted inside each CV fold (no leakage). Hyper-parameters are tuned with 5-fold stratified CV on the training split using **F1 of the spam class** (accuracy is misleading with ~13 % spam). The test set is used once.

In [ ]:
def metrics_dict(y, prob, thr=0.5):
    pred = (np.asarray(prob) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return dict(accuracy=float(accuracy_score(y, pred)), precision=float(precision_score(y, pred)),
                recall=float(recall_score(y, pred)), f1=float(f1_score(y, pred)),
                roc_auc=float(roc_auc_score(y, prob)), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))

def record(name, prob, **extra):
    m = metrics_dict(test_df.y.values, prob)
    pred = (np.asarray(prob) >= 0.5).astype(int)
    err = test_df.assign(pred=pred, prob=np.asarray(prob))
    fp = err[(err.pred == 1) & (err.y == 0)][["text", "prob"]].round(3)
    fn = err[(err.pred == 0) & (err.y == 1)][["text", "prob"]].round(3)
    RESULTS[name] = dict(metrics=m, false_positives=fp.to_dict("records"), false_negatives=fn.to_dict("records"), **extra)
    return pred

def plot_cm(name, ax):
    m = RESULTS[name]["metrics"]
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")

def avg_latency_ms(fn, n_rep=3):
    best = 1e9
    for _ in range(n_rep):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return 1000 * best / len(test_df)

nb_pipe = Pipeline([("tfidf", TfidfVectorizer(sublinear_tf=True)), ("clf", MultinomialNB())])
nb_grid = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [1, 2], "clf__alpha": [0.01, 0.05, 0.1, 0.5, 1.0]}
lr_pipe = Pipeline([("tfidf", TfidfVectorizer(sublinear_tf=True)), ("clf", LogisticRegression(max_iter=3000))])
lr_grid = {"tfidf__ngram_range": [(1, 1), (1, 2)], "tfidf__min_df": [1, 2], "clf__C": [1, 10, 100], "clf__class_weight": [None, "balanced"]}

fitted = {}
for name, pipe, grid in (("Naive Bayes", nb_pipe, nb_grid), ("Logistic Regression", lr_pipe, lr_grid)):
    t0 = time.perf_counter()
    gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=-1).fit(train_df.clean, train_df.y)
    train_time = time.perf_counter() - t0
    fitted[name] = gs.best_estimator_
    prob = gs.best_estimator_.predict_proba(test_df.clean)[:, 1]
    lat = avg_latency_ms(lambda: gs.best_estimator_.predict([preprocess(t, **PREP) for t in test_df.text]))
    V = len(gs.best_estimator_.named_steps["tfidf"].vocabulary_)
    n_params = 2 * V if name == "Naive Bayes" else V + 1   # NB: class-conditional log-probs; LR: weights + bias
    record(name, prob, best_params={k: str(v) for k, v in gs.best_params_.items()}, cv_f1=float(gs.best_score_),
           train_time_s=round(train_time, 2), latency_ms_per_msg=round(lat, 4), n_parameters=n_params)
    print(f"{name}: best CV F1 = {gs.best_score_:.4f} | params = {gs.best_params_}")

In [ ]:
res_tab = pd.DataFrame({n: RESULTS[n]["metrics"] for n in ("Naive Bayes", "Logistic Regression")}).T.round(4)
display(res_tab)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for a, n in zip(ax, ("Naive Bayes", "Logistic Regression")): plot_cm(n, a)
plt.tight_layout(); plt.savefig("figures/t3_confusion_matrices.png", dpi=200); plt.show()

for n in ("Naive Bayes", "Logistic Regression"):
    print(f"\n===== {n} =====")
    pred = (fitted[n].predict_proba(test_df.clean)[:, 1] >= 0.5).astype(int)
    print(classification_report(test_df.y, pred, target_names=["ham", "spam"], digits=4))
    print("FALSE POSITIVES (ham predicted as spam):"); display(pd.DataFrame(RESULTS[n]["false_positives"]))
    print("FALSE NEGATIVES (spam predicted as ham):"); display(pd.DataFrame(RESULTS[n]["false_negatives"]))

# most influential features of the logistic regression
lr = fitted["Logistic Regression"]
names = np.array(lr.named_steps["tfidf"].get_feature_names_out()); coef = lr.named_steps["clf"].coef_[0]
top_spam = [(names[i], round(float(coef[i]), 2)) for i in np.argsort(-coef)[:12]]
top_ham  = [(names[i], round(float(coef[i]), 2)) for i in np.argsort(coef)[:12]]
display(pd.DataFrame({"towards spam": [f"{w} ({c})" for w, c in top_spam], "towards ham": [f"{w} ({c})" for w, c in top_ham]}))
RESULTS["lr_top_features"] = dict(spam=top_spam, ham=top_ham)

# Task 4 – Sequence-Based Classification (BiLSTM)

The LSTM reads the message as an ordered sequence of word embeddings. It uses **lightly cleaned text** (same regex placeholders + lower-casing, but *no* stop-word removal / stemming and `!` `?` kept as tokens) because a sequence model needs the original word order and function words.

* Embedding 100-d (trained from scratch) → Bi-LSTM (64 units per direction) → dropout → linear output.
* Loss: weighted binary cross-entropy (`pos_weight` = ham/spam ratio) to handle imbalance.
* Early stopping on **validation F1**; the best epoch is restored. Test set is used once.
* Same threshold (0.5) as every other model.

In [ ]:
import torch, torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

def light_tokens(text): return re.findall(r"[a-z]+|[!?]", normalise(text))
cnt = Counter(w for t in train_df.text for w in light_tokens(t))
itos = ["<pad>", "<unk>"] + [w for w, c in cnt.most_common(10000) if c >= 2]
stoi = {w: i for i, w in enumerate(itos)}
lens = [len(light_tokens(t)) for t in train_df.text]
MAX_LEN = int(min(np.percentile(lens, 98), 100))
print(f"vocab size = {len(itos)} | max_len = {MAX_LEN} (98th percentile of token lengths)")

def encode(texts):
    ids = np.zeros((len(texts), MAX_LEN), dtype=np.int64); L = np.ones(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        toks = light_tokens(t)[:MAX_LEN]; L[i] = max(len(toks), 1)
        for j, w in enumerate(toks): ids[i, j] = stoi.get(w, 1)
    return torch.tensor(ids), torch.tensor(L)

def make_loader(texts, y, bs=64, shuffle=False):
    X, L = encode(list(texts))
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(TensorDataset(X, L, torch.tensor(np.asarray(y), dtype=torch.float32)), batch_size=bs, shuffle=shuffle, generator=g)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab, emb=100, hid=64, drop=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(drop); self.fc = nn.Linear(2 * hid, 1)
    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(h)).squeeze(1)

@torch.no_grad()
def lstm_probs(model, loader):
    model.eval(); out = []
    for X, L, _ in loader:
        out.append(torch.sigmoid(model(X.to(device), L)).cpu().numpy())
    return np.concatenate(out)

tr_loader  = make_loader(train_df.text, train_df.y, shuffle=True)
val_loader = make_loader(val_df.text, val_df.y)
te_loader  = make_loader(test_df.text, test_df.y)

model = BiLSTMClassifier(len(itos)).to(device)
n_params_lstm = sum(p.numel() for p in model.parameters())
pos_w = torch.tensor((train_df.y == 0).sum() / (train_df.y == 1).sum(), dtype=torch.float32, device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

history, best_f1, best_state, patience, bad = [], -1, None, 4, 0
t0 = time.perf_counter()
for epoch in range(1, 31):
    model.train(); tot = 0
    for X, L, y in tr_loader:
        X, y = X.to(device), y.to(device)
        opt.zero_grad(); loss = loss_fn(model(X, L), y); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tot += loss.item() * len(y)
    vp = lstm_probs(model, val_loader); vf1 = f1_score(val_df.y, vp >= 0.5)
    history.append(dict(epoch=epoch, train_loss=tot / len(train_df), val_f1=float(vf1)))
    print(f"epoch {epoch:2d} | train loss {tot/len(train_df):.4f} | val F1 {vf1:.4f}")
    if vf1 > best_f1: best_f1, best_state, bad = vf1, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        bad += 1
        if bad >= patience: print("early stopping"); break
lstm_train_time = time.perf_counter() - t0
model.load_state_dict(best_state)

prob_lstm = lstm_probs(model, te_loader)
lat = avg_latency_ms(lambda: lstm_probs(model, make_loader(test_df.text, test_df.y)))
record("BiLSTM", prob_lstm, best_val_f1=float(best_f1), history=history, train_time_s=round(lstm_train_time, 2),
       latency_ms_per_msg=round(lat, 4), n_parameters=int(n_params_lstm), vocab_size=len(itos), max_len=MAX_LEN)

h = pd.DataFrame(history); fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(h.epoch, h.train_loss, label="train loss"); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax2 = ax.twinx(); ax2.plot(h.epoch, h.val_f1, color="C1", label="val F1"); ax2.set_ylabel("val F1"); ax2.grid(False)
plt.title("BiLSTM training"); plt.tight_layout(); plt.savefig("figures/t4_lstm_training.png", dpi=200); plt.show()
print({k: round(v, 4) for k, v in RESULTS["BiLSTM"]["metrics"].items()})
print("FALSE POSITIVES:"); display(pd.DataFrame(RESULTS["BiLSTM"]["false_positives"]))
print("FALSE NEGATIVES:"); display(pd.DataFrame(RESULTS["BiLSTM"]["false_negatives"]))

### 4.1 Does word order matter? (word-shuffle test)

To check whether the LSTM actually exploits sequence information, the words of every **test** message are randomly shuffled. A bag-of-words model is order-invariant, so its predictions should barely change. If the LSTM relied on order, its score would drop clearly.

In [ ]:
rs = np.random.RandomState(SEED)
def shuffle_words(t):
    w = t.split(); rs.shuffle(w); return " ".join(w)
shuf = test_df.text.map(shuffle_words)

def f1_on(texts, kind):
    if kind == "lstm":
        return f1_score(test_df.y, lstm_probs(model, make_loader(texts, test_df.y)) >= 0.5)
    return f1_score(test_df.y, fitted[kind].predict([preprocess(t, **PREP) for t in texts]))

shuffle_test = {}
for kind, label in (("Naive Bayes", "Naive Bayes"), ("Logistic Regression", "Logistic Regression"), ("lstm", "BiLSTM")):
    a, b = f1_on(test_df.text, kind), f1_on(shuf, kind)
    shuffle_test[label] = dict(f1_original=float(a), f1_shuffled=float(b), drop=float(a - b))
display(pd.DataFrame(shuffle_test).T.round(4))
RESULTS["word_shuffle_test"] = shuffle_test

# Task 5 – Transformer-Based Classification (DistilBERT fine-tuning)

`distilbert-base-uncased` (a distilled BERT, 66 M parameters) is fine-tuned end-to-end on the **raw** SMS text (its own WordPiece tokenizer – no manual cleaning). 3 epochs, AdamW (lr 2e-5), linear warm-up/decay, class-weighted cross-entropy, mixed precision on GPU. The epoch with the best validation F1 is kept.
To use the full BERT instead change `MODEL_NAME` to `bert-base-uncased`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL_NAME = "distilbert-base-uncased"
EPOCHS, BATCH, LR, BERT_MAX_LEN = 3, 32, 2e-5, 128
set_seed()
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
n_params_bert = sum(p.numel() for p in bert.parameters())

def collate(batch):
    texts, labels = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=BERT_MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels); return enc
def bert_loader(df_, shuffle=False, bs=BATCH):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(list(zip(df_.text.tolist(), df_.y.tolist())), batch_size=bs, shuffle=shuffle, collate_fn=collate, generator=g)

@torch.no_grad()
def bert_probs(m, loader):
    m.eval(); out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        out.append(torch.softmax(m(**b).logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

b_tr, b_val, b_te = bert_loader(train_df, True), bert_loader(val_df), bert_loader(test_df)
use_amp = device.type == "cuda"
w = torch.tensor([1.0, float((train_df.y == 0).sum() / (train_df.y == 1).sum())], device=device)
loss_fn_b = nn.CrossEntropyLoss(weight=w)
opt_b = torch.optim.AdamW(bert.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(b_tr)
sched = get_linear_schedule_with_warmup(opt_b, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

hist_b, best_f1_b, best_state_b = [], -1, None
t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    bert.train(); tot = 0
    for batch in b_tr:
        batch = {k: v.to(device) for k, v in batch.items()}; labels = batch.pop("labels")
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = bert(**batch).logits
        loss = loss_fn_b(logits.float(), labels)
        opt_b.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt_b)
        nn.utils.clip_grad_norm_(bert.parameters(), 1.0); scaler.step(opt_b); scaler.update(); sched.step()
        tot += loss.item() * len(labels)
    vp = bert_probs(bert, b_val); vf1 = f1_score(val_df.y, vp >= 0.5)
    hist_b.append(dict(epoch=epoch, train_loss=tot / len(train_df), val_f1=float(vf1)))
    print(f"epoch {epoch} | train loss {tot/len(train_df):.4f} | val F1 {vf1:.4f}")
    if vf1 > best_f1_b: best_f1_b, best_state_b = vf1, {k: v.detach().clone() for k, v in bert.state_dict().items()}
bert_train_time = time.perf_counter() - t0
bert.load_state_dict(best_state_b)

prob_bert = bert_probs(bert, b_te)
lat = avg_latency_ms(lambda: bert_probs(bert, bert_loader(test_df)), n_rep=2)
record("DistilBERT", prob_bert, model_name=MODEL_NAME, best_val_f1=float(best_f1_b), history=hist_b,
       train_time_s=round(bert_train_time, 2), latency_ms_per_msg=round(lat, 4), n_parameters=int(n_params_bert), device=str(device))
print({k: round(v, 4) for k, v in RESULTS["DistilBERT"]["metrics"].items()})
print("FALSE POSITIVES:"); display(pd.DataFrame(RESULTS["DistilBERT"]["false_positives"]))
print("FALSE NEGATIVES:"); display(pd.DataFrame(RESULTS["DistilBERT"]["false_negatives"]))

# Final comparison and deployment recommendation

In [ ]:
order = ["Naive Bayes", "Logistic Regression", "BiLSTM", "DistilBERT"]
comp = pd.DataFrame({n: {**{k: RESULTS[n]["metrics"][k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc", "fp", "fn")},
                         "train_time_s": RESULTS[n]["train_time_s"], "latency_ms_per_msg": RESULTS[n]["latency_ms_per_msg"],
                         "parameters": RESULTS[n]["n_parameters"]} for n in order}).T
display(comp.round(4))
RESULTS["comparison"] = comp.round(5).to_dict("index")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
comp[["precision", "recall", "f1"]].plot.bar(ax=ax[0], rot=0, ylim=(0.8, 1.0)); ax[0].set_title("Spam-class metrics on the test set")
for a in ax[0].containers: ax[0].bar_label(a, fmt="%.3f", fontsize=7)
comp[["fp", "fn"]].plot.bar(ax=ax[1], rot=0, color=["#E6A23C", "#C44E52"]); ax[1].set_title("Test errors (FP = ham→spam, FN = spam→ham)")
for a in ax[1].containers: ax[1].bar_label(a)
plt.tight_layout(); plt.savefig("figures/t5_model_comparison.png", dpi=200); plt.show()

# which messages are hard for every model?
probs = {"Naive Bayes": fitted["Naive Bayes"].predict_proba(test_df.clean)[:, 1],
         "Logistic Regression": fitted["Logistic Regression"].predict_proba(test_df.clean)[:, 1],
         "BiLSTM": prob_lstm, "DistilBERT": prob_bert}
wrong = {n: set(np.where((p >= 0.5).astype(int) != test_df.y.values)[0]) for n, p in probs.items()}
all_wrong = set.intersection(*wrong.values()); any_wrong = set.union(*wrong.values())
print(f"Misclassified by ALL four models: {len(all_wrong)} | by at least one: {len(any_wrong)}")
hard = test_df.iloc[sorted(all_wrong)][["label", "text"]]; display(hard)
RESULTS["error_overlap"] = dict(all_models=len(all_wrong), at_least_one=len(any_wrong), hard_messages=hard.to_dict("records"),
                                per_model={n: len(s) for n, s in wrong.items()})

In [ ]:
def _clean(o):
    if isinstance(o, dict):  return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o
with open("results.json", "w") as fh: json.dump(_clean(RESULTS), fh, indent=1, ensure_ascii=False)

import shutil; shutil.make_archive("figures", "zip", "figures")
print("Saved: results.json and figures.zip  (download both from the Colab file browser)")
try:
    from google.colab import files
    files.download("results.json"); files.download("figures.zip")
except Exception:
    pass